In [5]:
# Google Colab Setup
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    !pip install datasets sentencepiece
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: Change to your project directory if needed
    # import os
    # os.chdir('/content/drive/MyDrive/NLP/Project_A3/A3_Burmese_English_Puffer')
except ImportError:
    IN_COLAB = False
    print("Running Locally")

Running in Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# German-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: German (de) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture.
We use the **Opus-100** dataset for German-English parallel data.
We use **SentencePiece** for subword tokenization.


In [6]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sentencepiece as spm

# Check for GPU
device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f"Using device: {device}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Using device: cuda


## 2. Data Loading (Opus-100)
Loading German-English pairs from Opus-100.

In [7]:
print("Loading Opus-100 Dataset (German-English)...")

data = []
try:
    # Opus-100 has 'de-en' or 'en-de'
    dataset = load_dataset("opus100", "de-en", split="train+validation+test")
    print(f"Loaded {len(dataset)} sentences from Opus-100 dataset.")

    for item in dataset:
        if 'translation' in item:
            # 'de' is the language code for German
            if 'de' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'de': item['translation']['de'],
                    'en': item['translation']['en']
                })

    # Limit to manageable size for this project
    if len(data) > 50000:
        import random
        random.shuffle(data)
        data = data[:50000]
        print("Subsampled to 50,000 examples for efficiency.")

    print(f"Extracted {len(data)} German-English pairs.")
except Exception as e:
    print(f"Error loading from HF: {e}")

Loading Opus-100 Dataset (German-English)...
Loaded 1004000 sentences from Opus-100 dataset.
Subsampled to 50,000 examples for efficiency.
Extracted 50000 German-English pairs.


In [8]:
df = pd.DataFrame(data)
print(df.head())

df = df.dropna(subset=['de', 'en'])
df['de'] = df['de'].astype(str)
df['en'] = df['en'].astype(str)
df = df[df['de'].str.strip() != '']
df = df[df['en'].str.strip() != '']

                                                  de  \
0                    Offenbar werde ich verdächtigt.   
1                                        Tielt +17°C   
2                                    Wie geht's dir?   
3  Zu ihm verhalten sich die Farben (guasch, temp...   
4                                              -Was?   

                                                  en  
0                         Apparently, I'm a suspect.  
1                                     Tucupido +28°C  
2                                       How are you?  
3  Paints concern them (gouache, distemper, poliv...  
4                                 You can't mean it!  


## 3. Tokenization

In [9]:
# Save texts to files
with open('train_de.txt', 'w', encoding='utf-8') as f:
    for line in df['de']: f.write(line + '\n')

with open('train_en_de.txt', 'w', encoding='utf-8') as f:
    for line in df['en']: f.write(line + '\n')

# Train SentencePiece models
vocab_size = 8000
model_type = 'bpe'

print("Training German Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_de.txt',
    model_prefix='spm_de',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer (for German pair)...")
spm.SentencePieceTrainer.train(
    input='train_en_de.txt',
    model_prefix='spm_en_de',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

sp_src = spm.SentencePieceProcessor(model_file='spm_de.model')
sp_trg = spm.SentencePieceProcessor(model_file='spm_en_de.model')

Training German Tokenizer...
Training English Tokenizer (for German pair)...


## 4. Dataset & Model

In [10]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['de']
        trg_text = self.data.iloc[idx]['en']
        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_pad, trg_pad

train_dataset = TranslationDataset(df, sp_src, sp_trg)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 d_model=256, nhead=4, num_encoder_layers=2,
                 num_decoder_layers=2, dim_feedforward=512, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg):
        src_key_padding_mask = (src == self.pad_idx)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        output = self.transformer(src=src_emb, tgt=trg_emb, tgt_mask=trg_mask, src_key_padding_mask=src_key_padding_mask)
        return self.fc_out(output)

In [12]:
model = TransformerModel(vocab_size, vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Starting Training...")
for epoch in range(10): # 10 Epochs for demo (Opus-100 is large)
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(train_loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        output = output.contiguous().view(-1, output.shape[-1])
        trg = trg[:, 1:].contiguous().view(-1)
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        if i % 100 == 0: print(f"Step {i}, Loss: {loss.item():.3f}")
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.3f}")

    # Save
    torch.save(model.state_dict(), 'transformer_model_de.pt')

Starting Training...
Step 0, Loss: 9.122
Step 100, Loss: 6.802
Step 200, Loss: 6.478
Step 300, Loss: 6.420
Step 400, Loss: 6.104
Step 500, Loss: 6.265
Step 600, Loss: 5.877
Step 700, Loss: 5.790
Epoch 1 Loss: 6.320
Step 0, Loss: 5.578
Step 100, Loss: 5.876
Step 200, Loss: 5.782
Step 300, Loss: 5.453
Step 400, Loss: 5.472
Step 500, Loss: 5.311
Step 600, Loss: 5.294
Step 700, Loss: 5.511
Epoch 2 Loss: 5.540
Step 0, Loss: 5.304
Step 100, Loss: 4.828
Step 200, Loss: 5.449
Step 300, Loss: 5.142
Step 400, Loss: 4.986
Step 500, Loss: 5.251
Step 600, Loss: 5.048
Step 700, Loss: 5.164
Epoch 3 Loss: 5.111
Step 0, Loss: 4.924
Step 100, Loss: 4.869
Step 200, Loss: 4.970
Step 300, Loss: 4.884
Step 400, Loss: 4.627
Step 500, Loss: 4.850
Step 600, Loss: 4.678
Step 700, Loss: 4.876
Epoch 4 Loss: 4.832
Step 0, Loss: 4.758
Step 100, Loss: 4.387
Step 200, Loss: 4.616
Step 300, Loss: 4.687
Step 400, Loss: 4.621
Step 500, Loss: 4.487
Step 600, Loss: 4.673
Step 700, Loss: 4.743
Epoch 5 Loss: 4.632
Step 0, L

In [13]:
# Copy to app
import shutil
os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model_de.pt', 'app/models/transformer_model_de.pt')
shutil.copy('spm_de.model', 'app/models/spm_de.model')
shutil.copy('spm_en_de.model', 'app/models/spm_en_de.model')
print("Models copied to app/models/")

Models copied to app/models/
